# Phase 5 Reproducibility

This notebook is the Phase-5 analysis entrypoint. It loads the committed manifest, renders the same transcript as the CLI report command, and prepares the table frames without fabricating empirical rows while Stage-7 recordings and experiment ledgers remain pending.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
backend_path = repo_root / "backend"
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

manifest_path = repo_root / "backend" / "experiments" / "manifests" / "phase5_pending_real_data.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest["status"], manifest["pending_external_inputs"]

## Manifest-Rendered Transcript

The Markdown report is rendered through the same manifest path used by `backend/scripts/render_phase5_report.py`, including validation of optional result artifacts and the manifest-declared `excluded_run_names` list.

In [ ]:
from scripts.render_phase5_report import render_report_from_manifest

report_markdown = render_report_from_manifest(manifest_path, repo_root=repo_root)
print(report_markdown[:1200])

## Per-Family Leaderboard Table

This table is built from `backend/experiments/*.jsonl` through `src.simulation.leaderboard.build_family_leaderboards`. Manifest-declared non-empirical runs are excluded here as well, so local smoke ledgers do not become research evidence.

In [ ]:
from src.simulation.experiment_tracker import ExperimentTracker
from src.simulation.leaderboard import build_family_leaderboards

runs_dir = repo_root / manifest["runs_dir"]
excluded_run_names = set(manifest.get("excluded_run_names", []))
leaderboards = build_family_leaderboards(ExperimentTracker(runs_dir))
leaderboard_rows = [
    entry.model_dump()
    for entries in leaderboards.values()
    for entry in entries
    if entry.run_name not in excluded_run_names
]
leaderboard_df = pd.DataFrame(leaderboard_rows)
leaderboard_columns = [
    "family",
    "run_name",
    "search_strategy",
    "held_out_accent",
    "mean_shape_distance",
    "lookup_ratio",
    "simplicity_score",
    "candidate_id",
]
if leaderboard_df.empty:
    leaderboard_df = pd.DataFrame(columns=leaderboard_columns)
leaderboard_df.loc[:, [column for column in leaderboard_columns if column in leaderboard_df.columns]]

## Table Sources

- Per-family leaderboards: `src.simulation.leaderboard.build_family_leaderboards` over `backend/experiments/*.jsonl`.
- Cross-accent tables: `src.simulation.leave_one_accent_out.evaluate_leave_one_accent_out` result JSON referenced by the manifest.
- Browser live-loop evidence: `src.models.live_loop_evidence.LiveLoopEvidence` JSON referenced by the manifest.
- Negative-results transcript: `src.simulation.negative_results.render_negative_results_report` through `scripts.render_phase5_report.render_report_from_manifest`.

Do not fill these tables with synthetic examples in the final writeup; use real experiment ledgers only.

In [ ]:
for key in ("families", "strategies", "distance_metrics", "excluded_run_names"):
    print(f"{key}: {', '.join(manifest.get(key, []))}")